In [4]:
import re
import math
from typing import Dict
import pandas as pd
import numpy as np

NA = 6.022_140_76e23      
ANG3_PER_CM3 = 1e24       
R_E_A = 2.81794e-5       
LAMBDA_A = 1.23984        


_NUM = r'(?:\d*\.?\d+)'  

def parse_formula(formula: str) -> Dict[str, float]:
   
    if not formula or not str(formula).strip():
        raise ValueError("Empty formula.")
    s = str(formula).replace(" ", "").replace("\u200b", "")
    parts = re.split(r"[·•.]", s)  

    def read_number(txt, i):
        m = re.match(_NUM, txt[i:])
        if not m:
            return 1.0, i
        val = float(m.group(0))
        return val, i + len(m.group(0))

    total: Dict[str, float] = {}
    for part in parts:
        if not part:
            continue
        
        mlead = re.match(rf"^{_NUM}(?=[A-Z(])", part)
        frag_mult = float(mlead.group(0)) if mlead else 1.0
        frag = part[len(mlead.group(0)):] if mlead else part

        stack = [dict()]
        i = 0
        while i < len(frag):
            ch = frag[i]
            if ch == '(':
                stack.append(dict())
                i += 1
            elif ch == ')':
                i += 1
                mult, i = read_number(frag, i)
                inner = stack.pop()
                for k, v in inner.items():
                    stack[-1][k] = stack[-1].get(k, 0.0) + v * mult
            elif ch.isupper():
                sym = ch
                i += 1
                if i < len(frag) and frag[i].islower():
                    sym += frag[i]
                    i += 1
                mult, i = read_number(frag, i)
                stack[-1][sym] = stack[-1].get(sym, 0.0) + mult
            elif ch.isdigit() or ch == '.':
                
                _, i = read_number(frag, i)
            else:
                raise ValueError(f"Unexpected char '{ch}' in '{formula}' near '{frag[i:]}'")

        top = stack.pop()
        for k, v in top.items():
            total[k] = total.get(k, 0.0) + frag_mult * v

    return total


def alpha_S6(eps_inf: float, Eg_eV: float, sum_Nf1_Ainv3: float) -> tuple[float, float]:
   
    if eps_inf is None or Eg_eV is None or sum_Nf1_Ainv3 is None:
        return (float("nan"), float("nan"))
    if eps_inf <= 1 or Eg_eV <= 0 or sum_Nf1_Ainv3 <= 0:
        return (float("nan"), float("nan"))

    arg = (R_E_A * (LAMBDA_A ** 2) * sum_Nf1_Ainv3) / (
        math.pi * (eps_inf - 1.0) * (Eg_eV ** 1.2)
    )
    if arg <= 0 or not np.isfinite(arg):
        return (float("nan"), float("nan"))

    alpha = (-0.36 * math.log10(arg)) - 0.545
    return (alpha, arg)


def compute_alpha(
    alpha_input_path="Alpha-Non Layered TMHs-input.xlsx",
    elements_info_path="Elements Info.xlsx",
    out_path="Alpha-Non Layered TMHs-output.xlsx",
    include_notes=True
):
    
    df = pd.read_excel(alpha_input_path)
    elem = pd.read_excel(elements_info_path)

   
    req_alpha_cols = ["Formula", "eps_inf", "Eg", "Density"]
    req_elem_cols = ["Element", "f1", "Molarmass"]
    for c in req_alpha_cols:
        if c not in df.columns:
            raise ValueError(f"Missing column '{c}' in {alpha_input_path}")
    for c in req_elem_cols:
        if c not in elem.columns:
            raise ValueError(f"Missing column '{c}' in {elements_info_path}")

    
    def norm_sym(x):
        s = str(x).strip().replace("*", "").replace("\u200b", "")
        return s[0].upper() + s[1:].lower() if s else s

    elem["sym"] = elem["Element"].apply(norm_sym)
    f1_map = {
        row["sym"]: float(row["f1"])
        for _, row in elem.iterrows()
        if pd.notna(row["sym"]) and pd.notna(row["f1"])
    }
    mass_map = {
        row["sym"]: float(row["Molarmass"])
        for _, row in elem.iterrows()
        if pd.notna(row["sym"]) and pd.notna(row["Molarmass"])
    }

  
    out_rows = []
    for _, r in df.iterrows():
        formula = str(r["Formula"]).strip()
        eps_inf = pd.to_numeric(r["eps_inf"], errors="coerce")
        Eg_eV   = pd.to_numeric(r["Eg"],       errors="coerce")
        rho     = pd.to_numeric(r["Density"],  errors="coerce")  

        row = dict(r)  

       
        row.update({
            "MolarMass_g_mol": np.nan,  
            "sum_sf1": np.nan,
            "fu_per_cm3": np.nan,
            "fu_per_A3": np.nan,
            "sum_Nf1_A^-3": np.nan,
            "log_argument": np.nan,
            "alpha_S6": np.nan
        })
        if include_notes:
            row["note"] = ""

        
        if any(pd.isna(x) for x in [eps_inf, Eg_eV, rho]) or rho <= 0:
            if include_notes:
                row["note"] = "Missing/invalid eps_inf, Eg, or Density"
            out_rows.append(row)
            continue

       
        try:
            stoich = parse_formula(formula)
        except Exception as e:
            if include_notes:
                row["note"] = f"Parse error: {e}"
            out_rows.append(row)
            continue

       
        M = 0.0
        sum_sf1 = 0.0
        missing = []
        for el, s in stoich.items():
            key = norm_sym(el)
            Mi  = mass_map.get(key, np.nan)
            f1i = f1_map.get(key,  np.nan)
            if np.isnan(Mi) or np.isnan(f1i):
                missing.append(key)
            else:
                M += s * Mi
                sum_sf1 += s * f1i

        if missing or M <= 0 or sum_sf1 <= 0:
            if include_notes:
                row["note"] = f"Missing element data: {', '.join(missing)}"
            out_rows.append(row)
            continue

       
        fu_per_cm3 = (rho / M) * NA
        fu_per_A3  = fu_per_cm3 / ANG3_PER_CM3

        
        sum_Nf1_Ainv3 = fu_per_A3 * sum_sf1

       
        alpha, log_arg = alpha_S6(float(eps_inf), float(Eg_eV), sum_Nf1_Ainv3)

       
        row.update({
            "MolarMass_g_mol": M,            
            "sum_sf1": sum_sf1,
            "fu_per_cm3": fu_per_cm3,
            "fu_per_A3": fu_per_A3,
            "sum_Nf1_A^-3": sum_Nf1_Ainv3,
            "log_argument": log_arg,
            "alpha_S6": alpha
        })
        out_rows.append(row)

    df_out = pd.DataFrame(out_rows)

    
    front = [c for c in ["Formula", "MolarMass_g_mol"] if c in df_out.columns]
    rest = [c for c in df_out.columns if c not in front]
    df_out = df_out[front + rest]

    df_out.to_excel(out_path, index=False)
    print(f"Saved: {out_path}")


if __name__ == "__main__":
    compute_alpha(
        alpha_input_path="Alpha-Non Layered TMHs-input.xlsx",
        elements_info_path="Elements Info.xlsx",
        out_path="Alpha-NonLayered TMHs-output.xlsx",
        include_notes=True
    )


Saved: Alpha-NonLayered TMHs-output.xlsx
